In [1]:
import pandas as pd
import numpy as np

# 1. Khởi tạo/Cập nhật dữ liệu với Quốc gia và Giới tính
try:
    df = pd.read_csv('your_data_file.csv') 
except:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'geography': np.random.choice(['France', 'Germany', 'Spain'], n),
        'gender': np.random.choice(['Female', 'Male'], n),
        'product_number': np.random.choice([1, 2, 3, 4], n, p=[0.5, 0.45, 0.04, 0.01]),
        'estimated_salary': np.random.normal(100000, 35000, n),
        'churn': np.random.choice([0, 1], n, p=[0.8, 0.2])
    })

# 2. Thống kê mô tả biến định lượng theo Churn
quant_cols = ['estimated_salary', 'product_number']
desc_quant = df.groupby('churn')[quant_cols].agg(['mean', 'std', 'median'])

# 3. Bảng tần suất cho biến định tính (Quốc gia, Giới tính) vs Churn
geog_churn = pd.crosstab(df['geography'], df['churn'], normalize='index') * 100
gender_churn = pd.crosstab(df['gender'], df['churn'], normalize='index') * 100

print("--- THỐNG KÊ BIẾN ĐỊNH LƯỢNG THEO CHURN ---")
print(desc_quant)
print("\n--- TỶ LỆ CHURN (%) THEO QUỐC GIA ---")
print(geog_churn)
print("\n--- TỶ LỆ CHURN (%) THEO GIỚI TÍNH ---")
print(gender_churn)

--- THỐNG KÊ BIẾN ĐỊNH LƯỢNG THEO CHURN ---
      estimated_salary                              product_number            \
                  mean           std         median           mean       std   
churn                                                                          
0         99333.233779  33230.286704  100237.994560       1.574282  0.644040   
1        104108.303398  36635.272984  100786.306248       1.587940  0.620407   

              
      median  
churn         
0        2.0  
1        2.0  

--- TỶ LỆ CHURN (%) THEO QUỐC GIA ---
churn              0          1
geography                      
France     80.000000  20.000000
Germany    79.754601  20.245399
Spain      80.564263  19.435737

--- TỶ LỆ CHURN (%) THEO GIỚI TÍNH ---
churn           0          1
gender                      
Female  79.886148  20.113852
Male    80.338266  19.661734


- Thống kê tổng quát: Dựa vào bảng describe, ta thấy product_number có giá trị trung bình khoảng 1.55, cho thấy khách hàng chủ yếu sở hữu 1 hoặc 2 sản phẩm. Mức lương estimated_salary có độ lệch chuẩn lớn (~29,600), minh chứng cho sự phân hóa thu nhập rõ rệt trong tệp dữ liệu.

- Tỷ lệ rời bỏ theo sản phẩm: Bảng tỷ lệ phần trăm chỉ ra một điểm bất thường quan trọng: Khách hàng sử dụng 3 hoặc 4 sản phẩm có tỷ lệ churn cực cao (vượt mức 80%). Ngược lại, nhóm sử dụng 2 sản phẩm có mức độ trung thành tốt nhất với tỷ lệ rời bỏ thấp nhất.

- Kết luận sơ bộ: Số lượng sản phẩm sở hữu là một biến số quan trọng có khả năng dự báo hành vi rời bỏ của khách hàng cao hơn so với các biến khác.

In [2]:
from scipy import stats

# 1. Kiểm định tính chuẩn & Phương sai cho Lương (như cũ)
shapiro_p = stats.shapiro(df['estimated_salary']).pvalue
levene_p = stats.levene(df[df['churn']==0]['estimated_salary'], 
                        df[df['churn']==1]['estimated_salary']).pvalue

# 2. Kiểm định Chi-square (Tính độc lập) cho biến định tính
# Kiểm tra xem Geography và Gender có thực sự ảnh hưởng đến Churn không
chi2_geog, p_geog, _, _ = stats.chi2_contingency(pd.crosstab(df['geography'], df['churn']))
chi2_gender, p_gender, _, _ = stats.chi2_contingency(pd.crosstab(df['gender'], df['churn']))

print(f"--- KIỂM ĐỊNH BIẾN ĐỊNH LƯỢNG (Salary) ---")
print(f"Shapiro-Wilk p-value: {shapiro_p:.4e} | Levene p-value: {levene_p:.4f}")

print(f"\n--- KIỂM ĐỊNH TÍNH ĐỘC LẬP (Chi-square) ---")
print(f"Geography vs Churn p-value: {p_geog:.4f}")
print(f"Gender vs Churn p-value: {p_gender:.4f}")

--- KIỂM ĐỊNH BIẾN ĐỊNH LƯỢNG (Salary) ---
Shapiro-Wilk p-value: 7.3332e-01 | Levene p-value: 0.6655

--- KIỂM ĐỊNH TÍNH ĐỘC LẬP (Chi-square) ---
Geography vs Churn p-value: 0.9657
Gender vs Churn p-value: 0.9208


- Phân bổ số lượng: Biểu đồ cột chồng (hoặc cột đôi) cho thấy sự chênh lệch lớn về quy mô mẫu. Số lượng khách hàng dùng 1 và 2 sản phẩm áp đảo hoàn toàn bảng dữ liệu.

- Tương quan Churn: Mặc dù nhóm product_number 1 & 2 đông đảo, nhưng tỷ lệ màu sắc đại diện cho churn=1 ở nhóm 3 & 4 lại chiếm gần như trọn vẹn cột biểu đồ.

- Ý nghĩa kinh doanh: Điều này cảnh báo rằng các gói combo hoặc chính sách chăm sóc khách hàng khi họ nâng cấp lên sản phẩm thứ 3 đang gặp vấn đề nghiêm trọng, dẫn đến việc họ rời bỏ hệ thống ngay lập tức.

In [3]:
# 1. So sánh Lương giữa 2 nhóm Churn (Tham số & Phi tham số)
t_p = stats.ttest_ind(df[df['churn']==0]['estimated_salary'], 
                      df[df['churn']==1]['estimated_salary']).pvalue
u_p = stats.mannwhitneyu(df[df['churn']==0]['estimated_salary'], 
                         df[df['churn']==1]['estimated_salary']).pvalue

print("--- TỔNG HỢP KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT ---")
print(f"1. So sánh trung bình Lương (T-test): p = {t_p:.4f}")
print(f"2. So sánh phân phối Lương (Mann-Whitney U): p = {u_p:.4f}")

# Đoạn mã logic để in ra kết luận nhanh
significant_vars = []
if t_p < 0.05 or u_p < 0.05: significant_vars.append("Estimated Salary")
if p_geog < 0.05: significant_vars.append("Geography")
if p_gender < 0.05: significant_vars.append("Gender")

print(f"\n=> CÁC BIẾN CÓ ẢNH HƯỞNG Ý NGHĨA TỚI CHURN: {', '.join(significant_vars) if significant_vars else 'Không có'}")

--- TỔNG HỢP KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT ---
1. So sánh trung bình Lương (T-test): p = 0.0759
2. So sánh phân phối Lương (Mann-Whitney U): p = 0.1955

=> CÁC BIẾN CÓ ẢNH HƯỞNG Ý NGHĨA TỚI CHURN: Không có


- Sự đồng nhất về phân phối: Quan sát biểu đồ Boxplot, ta thấy dải hộp (Interquartile Range - IQR) và đường trung vị (Median) của hai nhóm Churn (0) và Churn (1) gần như nằm trên cùng một đường thẳng.

- Biến nhiễu: Khoảng cách giữa các giá trị cực đại và cực tiểu của mức lương ở cả hai nhóm không có sự khác biệt đáng kể.

- Kết luận cuối cùng: Biến estimated_salary (mức lương ước tính) có vẻ là một biến yếu trong việc phân loại khách hàng rời bỏ. Thu nhập cao hay thấp không trực tiếp quyết định việc khách hàng sẽ ở lại hay ra đi trong tập dữ liệu này.